In [0]:
spark.conf.set(
    "fs.azure.account.auth.type.testgen2strg.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.testgen2strg.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.testgen2strg.dfs.core.windows.net",
    dbutils.secrets.get(scope="testsecretscope", key="appid")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.testgen2strg.dfs.core.windows.net",
    dbutils.secrets.get(scope="testsecretscope", key="apppwd")
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.testgen2strg.dfs.core.windows.net",
    "https://login.microsoftonline.com/0c44fd98-6a6c-40f7-8cbe-a03159095bd8/oauth2/v2.0/token"
)

In [0]:
src_path = "abfss://global@testgen2strg.dfs.core.windows.net/silver/"
dest_path = "abfss://global@testgen2strg.dfs.core.windows.net/gold/"

In [0]:
dbutils.widgets.text('tablename','')
dbutils.widgets.text('cdate','')

In [0]:
sourcename = dbutils.widgets.get('tablename')
cdate = dbutils.widgets.get('cdate')

print(sourcename)
print(cdate)

In [0]:
src_final_path = src_path + sourcename + "/" + cdate
print(src_final_path)

dest_final_path = dest_path + "dim" + sourcename
print(dest_final_path)

In [0]:
df = spark.read.format("csv").option("header", True).load(src_final_path)

#df.show()

src_count = df.count()
print(src_count)

In [0]:
df.show()

In [0]:
df11 = spark.createDataFrame(
    [(2, '78654345'), (3, '67865467')],
    ['cid', 'cphone']
)

df11.show()

In [0]:
from pyspark.sql.functions import col

if sourcename == 'cust':
    df = df.filter(
        (col("cid").isNotNull()) &
        (col("cid") != "null")
    )

    df1 = df.alias('a').join(
        df11.alias('b'),
        col('a.cid').cast("long") == col('b.cid'),
        "inner"
    ).select('a.*', 'b.cphone')

    df1.show()

else:
    df1 = df

In [0]:
dest_count = df1.count()

In [0]:
df1.coalesce(1).write.mode("overwrite") \
    .format("csv") \
    .option("header", True) \
    .save(dest_final_path)

In [0]:
dbutils.notebook.exit(
    "source count:" + str(src_count) +
    " destination count:" + str(dest_count)
)